# Módulo F5 · Soluciones de la práctica guiada

## Repaso del flujo completo

In [ ]:
%%bash
# El proyecto sandbox de la práctica ya está construido. Recapitulamos: 
cd /tmp/f5-sandbox/analisis-tiendas 2>/dev/null || { echo "Recreando..."; exit 1; }

echo "=== historial ==="
git log --oneline

echo "=== ramas ==="
git branch -a

echo "=== archivos tracked ==="
git ls-files

## Ejercicio 1 · Repo propio (checklist)

Checklist de lo que debe tener tu repo real:

- [x] `git init` en una carpeta nueva (no en tu home)
- [x] `.gitignore` con `.venv/`, `__pycache__/`, `*.csv` grandes
- [x] `README.md` con: qué hace, cómo ejecutar, ejemplo
- [x] un `app.py` (o notebook) que funcione de arriba abajo
- [x] al menos 3 commits con mensajes descriptivos


## Ejercicio 2 · venv y requirements

In [ ]:
%%bash
cd /tmp/f5-sandbox/analisis-tiendas
if [ -d .venv ]; then
  .venv/bin/pip freeze 2>/dev/null | head -10
  echo "----"
  .venv/bin/pip list 2>/dev/null | grep -i pandas
else
  echo "(sandbox sin venv) — en tu máquina: pip freeze > requirements.txt"
fi

## Ejercicio 3 · rama y merge

In [ ]:
%%bash
cd /tmp/f5-sandbox/analisis-tiendas
# Si no lo hiciste en la práctica, aquí va el patrón:
git switch -c demo-merge 2>/dev/null || git switch demo-merge
echo "- Mejora demo en README" >> mejora.md
git add mejora.md
git commit -q -m "Demo: mejora en mejoras.md"
git switch main
git merge --no-edit demo-merge
git log --oneline | head -5

## Ejercicio 4 · revisar mensajes de commit

Abre `git log --oneline` y busca commits tipo "fix", "cambios" o "más cosas".

Ejemplo de mejora: `-m "Fix: corregir filtro de fechas en app.py (ignoraba pedidos de enero)"`.
La regla: **el mensaje explica el por qué**, no solo el qué.

## Ejercicio 5 · publicar en GitHub

Pasos (una vez, con tu cuenta):

1. Crea el repo en github.com → *New repository* → sin README inicial.
2. `git remote add origin git@github.com:tusuario/analisis-tiendas.git`
3. `git push -u origin main`
4. Opcional: activa la pestaña Issues y añade tu primer issue ("añadir tests").

Tu entregable para el módulo es el enlace al repo con: README, requirements, commits limpios y (idealmente) `clear outputs` en los notebooks.

---

## 5.7 · Tu primer día con un agente de código

Un agente de código (Claude Code, Cursor...) lee tu repositorio, edita archivos y ejecuta comandos. No sustituye lo que has aprendido en este módulo: lo **necesita** (Git para ver y deshacer, terminal para que ejecute tests, entorno para no romper nada).

En esta sección no llamamos a ningún agente real (no hace falta cuenta ni clave). Montamos lo que **tú** debes preparar antes de dejarle trabajar y practicamos lo que **tú** haces después: leer el diff. Un «agente simulado» (una función que aplica un parche) hará de agente.

In [ ]:
import re
import subprocess
import sys
import tempfile
from pathlib import Path

REPO = Path(tempfile.mkdtemp(prefix="f5-agente-"))   # carpeta temporal: no ensucia tu disco
(REPO / "tiendasur").mkdir()
(REPO / "tests").mkdir()

def sh(*args: str) -> int:
    """Ejecuta un comando en el repo de práctica y muestra su salida."""
    r = subprocess.run(list(args), cwd=REPO, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1500:])
    return r.returncode

def git(*args: str) -> int:
    return sh("git", "-c", "user.name=Alumno", "-c", "user.email=alumno@example.com", *args)

(REPO / "tiendasur" / "__init__.py").write_text("")
(REPO / "tiendasur" / "precios.py").write_text(
    "def precio_final(precio: float, descuento_pct: float = 0) -> float:\n"
    "    return round(precio * (1 - descuento_pct / 100), 2)\n"
)
(REPO / "tests" / "test_precios.py").write_text(
    "from tiendasur.precios import precio_final\n\n"
    "def test_sin_descuento():\n    assert precio_final(10) == 10\n\n"
    "def test_descuento_normal():\n    assert precio_final(10, 20) == 8\n"
)
# Instrucciones para el agente: cómo se prueba y qué NO se toca.
(REPO / "AGENTS.md").write_text("""# Instrucciones para agentes

## Comandos
- Tests: `python -m pytest -q`

## Reglas
- Todo cambio de comportamiento lleva su test. Los tests existentes NO se debilitan para que pasen.
- No escribas claves, tokens ni contraseñas en el código: se leen de variables de entorno.
- No instales dependencias nuevas sin avisar.
- Cambios pequeños: un objetivo por diff.
""")
(REPO / ".gitignore").write_text(".env\n__pycache__/\n.venv/\n")
git("init", "-q")
git("add", ".")
git("commit", "-q", "-m", "Base: precio_final con tests e instrucciones para agentes")
git("log", "--oneline")

### Paso 1 · Los tests son la especificación

Queremos una regla nueva: **el descuento máximo es del 50 %** y un descuento negativo o superior a 100 es un error de datos. Antes de pedir nada, escribimos el test: así el agente (y tú) sabéis cuándo está terminado.

In [ ]:
(REPO / "tests" / "test_precios.py").write_text(
    "import pytest\n"
    "from tiendasur.precios import precio_final\n\n"
    "def test_sin_descuento():\n    assert precio_final(10) == 10\n\n"
    "def test_descuento_normal():\n    assert precio_final(10, 20) == 8\n\n"
    "def test_descuento_maximo_50():\n    assert precio_final(10, 80) == 5\n\n"
    "def test_descuento_invalido():\n"
    "    with pytest.raises(ValueError):\n        precio_final(10, -5)\n"
    "    with pytest.raises(ValueError):\n        precio_final(10, 120)\n"
)
print("Tests ANTES del cambio (deben fallar: es la especificación sin cumplir):")
sh(sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--tb=no")

### Paso 2 · «El agente» propone un cambio

Una función hace de agente: aplica un parche. Un agente real hace lo mismo, pero decidiendo él el contenido. Mira el **diff** antes de aceptar nada.

In [ ]:
def agente_bueno() -> None:
    (REPO / "tiendasur" / "precios.py").write_text(
        "DESCUENTO_MAXIMO = 50\n\n"
        "def precio_final(precio: float, descuento_pct: float = 0) -> float:\n"
        "    if not 0 <= descuento_pct <= 100:\n"
        "        raise ValueError(f'descuento fuera de rango: {descuento_pct}')\n"
        "    descuento = min(descuento_pct, DESCUENTO_MAXIMO)\n"
        "    return round(precio * (1 - descuento / 100), 2)\n"
    )

agente_bueno()
git("diff", "--stat")
git("diff", "--", "tiendasur")

In [ ]:
print("Tests DESPUÉS del cambio:")
sh(sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider")

### Paso 3 · Revisar el diff como el de un compañero

Una parte de la revisión se puede automatizar. La función siguiente busca tres señales de alarma típicas en un diff generado por un agente: **tests debilitados** (líneas `assert` borradas), **secretos** añadidos y **cambios demasiado grandes**. No sustituye a leer el diff: lo complementa.

In [ ]:
PATRON_SECRETO = re.compile(r"(sk-[A-Za-z0-9_-]{16,}|AKIA[0-9A-Z]{16}|(?i:password|secret|token|api_key)\s*=\s*['\"][^'\"]{6,})")

def revisar_diff(max_lineas: int = 80) -> list[str]:
    """Devuelve avisos sobre el diff sin confirmar del repo de práctica."""
    subprocess.run(["git", "add", "-N", "."], cwd=REPO)          # que los archivos nuevos aparezcan en el diff
    diff = subprocess.run(["git", "diff"], cwd=REPO, capture_output=True, text=True).stdout
    avisos = []
    añadidas = [l[1:] for l in diff.splitlines() if l.startswith("+") and not l.startswith("+++")]
    borradas = [l[1:] for l in diff.splitlines() if l.startswith("-") and not l.startswith("---")]
    if any("assert" in l or "raises" in l for l in borradas):
        avisos.append("Se han BORRADO aserciones: ¿se está debilitando un test para que pase?")
    if any(PATRON_SECRETO.search(l) for l in añadidas):
        avisos.append("Posible SECRETO añadido al código: usa variables de entorno.")
    if len(añadidas) + len(borradas) > max_lineas:
        avisos.append(f"Diff grande ({len(añadidas) + len(borradas)} líneas): pide cambios más pequeños.")
    return avisos

print("Diff del agente bueno ->", revisar_diff() or "sin avisos")

### Paso 4 · Cuando el agente «hace trampa»

Un caso real y frecuente: ante un test que falla, el camino más corto es cambiar el test. Simulamos un agente que lo hace y además deja una clave de API (falsa) en el código.

In [ ]:
git("add", "-A")
git("commit", "-q", "-m", "Descuento máximo del 50 % con validación")      # aceptamos el cambio bueno

def agente_tramposo() -> None:
    (REPO / "tests" / "test_precios.py").write_text(
        "from tiendasur.precios import precio_final\n\n"
        "def test_sin_descuento():\n    assert precio_final(10) == 10\n"
    )                                                                      # ha borrado tests
    (REPO / "tiendasur" / "config.py").write_text('API_KEY = "sk-FALSA-0123456789abcdefghij"\n')

agente_tramposo()
for aviso in revisar_diff():
    print("AVISO:", aviso)
git("checkout", "--", "tests")                                             # deshacer: Git es tu red de seguridad
(REPO / "tiendasur" / "config.py").unlink()
git("reset", "-q")                                                         # quitar el intent-to-add
print("Estado tras deshacer:"); git("status", "--short")

### Ejercicios de 5.7

1. **Especificación primero.** TiendaSur quiere `envio_gratis(total: float) -> bool`: el envío es gratis a partir de 40 € (incluido). Escribe en `tests/test_envio.py` **solo el test** (tres casos: 39,99 / 40 / 100) y comprueba que falla.
2. **Pide el cambio.** Escribe, en una celda de texto o comentario, la petición que le harías a un agente: objetivo, archivos a tocar, criterio de aceptación y una restricción. Después implementa tú `tiendasur/envio.py` como lo haría el agente y comprueba que el test pasa.
3. **Amplía la revisión.** Añade a `revisar_diff` un aviso si el diff modifica `AGENTS.md` (un agente no debería cambiar sus propias reglas sin que lo veas).

In [ ]:
(REPO / "tests" / "test_envio.py").write_text(
    "from tiendasur.envio import envio_gratis\n\n"
    "def test_por_debajo():\n    assert envio_gratis(39.99) is False\n\n"
    "def test_umbral_incluido():\n    assert envio_gratis(40) is True\n\n"
    "def test_por_encima():\n    assert envio_gratis(100) is True\n"
)
(REPO / "tiendasur" / "envio.py").write_text("def envio_gratis(total: float) -> bool:\n    raise NotImplementedError\n")
print("Rojo (el test existe y falla porque aún no hay lógica):")
sh(sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--tb=no", "tests/test_envio.py")

In [ ]:
# Petición al agente (ejemplo):
# Objetivo: envío gratis desde 40 € (incluido).
# Archivos: solo tiendasur/envio.py; NO modifiques tests/test_envio.py.
# Aceptación: `python -m pytest -q` en verde.
# Restricción: sin dependencias nuevas; la constante del umbral con nombre.
(REPO / "tiendasur" / "envio.py").write_text(
    "UMBRAL_ENVIO_GRATIS = 40\n\n"
    "def envio_gratis(total: float) -> bool:\n    return total >= UMBRAL_ENVIO_GRATIS\n"
)
print("Verde:")
sh(sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider")

In [ ]:
_revisar_base = revisar_diff

def revisar_diff(max_lineas: int = 80) -> list[str]:
    avisos = _revisar_base(max_lineas)
    cambiados = subprocess.run(["git", "diff", "--name-only"], cwd=REPO, capture_output=True, text=True).stdout.split()
    if "AGENTS.md" in cambiados:
        avisos.append("El agente ha modificado AGENTS.md: sus propias reglas.")
    return avisos

with open(REPO / "AGENTS.md", "a") as f:
    f.write("- (regla añadida por el agente)\n")
print(revisar_diff())

**Qué te llevas:** un agente rinde en proporción a la estructura que le das (instrucciones, tests, Git) y tú sigues siendo quien responde del resultado. Si no sabes explicar un diff, no lo aceptes.